# 02. Interferogram formation core

This notebook reproduces the exact arithmetic that `InterferogramBuilder._compute_interferograms` applies to form an interferogram from a primary and a secondary SLC: deramp the secondary by the DEM phase, form the conjugate phasor against the primary, normalise the phasor to unit magnitude, and re-weight it by the clipped secondary amplitude. The real method reads SLCs from disk through PyRat; here the same expression is evaluated on a synthetic stack so the transform can be verified in isolation.

**Modules exercised:** pipelines.processing_pipeline.interferogram (InterferogramBuilder), configuration.processing_config (TomogramConfiguration.max_amplitude_clip)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## Reference: the formation expression

From `interferogram.py`, per secondary track the builder computes:

```
secondary_amplitude = clip(|secondary|, 0, max_amplitude_clip)
deramped_secondary  = secondary * exp(1j * dem_phase)
phasor              = primary * conj(deramped_secondary)
phasor             /= (|phasor| + 1e-30)
interferogram       = secondary_amplitude * phasor
```

The output therefore carries the interferometric phase in its argument and the clipped secondary amplitude in its magnitude.

## Build a small synthetic stack

We reuse the planting helper from notebook 01 so the interferometric phase has a known geometric origin.

In [ ]:
def synthetic_slc_stack(n_tracks, n_azimuth, n_range, kz, scatterers, rng, noise_sigma=0.05):
    azimuth_axis = np.arange(n_azimuth)
    range_axis   = np.arange(n_range)
    az_grid, rg_grid = np.meshgrid(azimuth_axis, range_axis, indexing="ij")

    reflectivity = np.zeros((n_azimuth, n_range), dtype=np.float32)
    heights      = np.zeros((n_azimuth, n_range), dtype=np.float32)

    for az_center, rg_center, sigma, amplitude, height in scatterers:
        blob          = amplitude * np.exp(-(((az_grid - az_center) ** 2 + (rg_grid - rg_center) ** 2) / (2.0 * sigma ** 2)))
        reflectivity += blob.astype(np.float32)
        heights       = np.where(blob > reflectivity * 0.5, height, heights).astype(np.float32)

    stack = np.empty((n_tracks, n_azimuth, n_range), dtype=np.complex64)
    for track_index in range(n_tracks):
        topographic_phase  = kz[track_index] * heights
        amplitude_field    = reflectivity
        complex_field      = amplitude_field * np.exp(1.0j * topographic_phase)
        speckle            = (rng.standard_normal(complex_field.shape) + 1.0j * rng.standard_normal(complex_field.shape))
        complex_field     += noise_sigma * speckle.astype(np.complex64)
        stack[track_index] = complex_field.astype(np.complex64)

    return stack, reflectivity, heights


In [ ]:
n_tracks, n_az, n_rg = 5, 48, 64
kz = np.linspace(0.0, 0.25, n_tracks)
scatterers = [(12, 20, 2.5, 1.0, 10.0), (30, 45, 2.5, 0.9, 45.0)]

stack, reflectivity, heights = synthetic_slc_stack(n_tracks, n_az, n_rg, kz, scatterers, RNG)
primary      = stack[0]
secondaries  = stack[1:]
print('primary    :', primary.shape)
print('secondaries:', secondaries.shape)

## Synthetic DEM phase

The pipeline removes a known reference (DEM) phase before forming the interferogram. We model a smooth range-dependent DEM phase per secondary, scaled by that track's kz, exactly as the topographic term would be.

In [ ]:
range_axis = np.arange(n_rg)
dem_height = 8.0 * np.sin(2.0 * np.pi * range_axis / n_rg)
dem_phase  = np.stack([kz[t + 1] * np.broadcast_to(dem_height, (n_az, n_rg)) for t in range(n_tracks - 1)])
print('dem_phase stack:', dem_phase.shape)

## Apply the exact formation expression

We replicate the builder arithmetic verbatim and compare an interferogram formed with the DEM removed against one formed without removing it.

In [ ]:
max_amplitude_clip = 1.25

def form_interferogram(primary, secondary, dem_phase):
    secondary_amplitude = np.clip(np.abs(secondary), 0.0, max_amplitude_clip)
    deramped_secondary  = secondary * np.exp(1.0j * dem_phase)
    phasor              = primary * np.conj(deramped_secondary)
    phasor             /= (np.abs(phasor) + 1e-30)
    return secondary_amplitude * phasor

track = 2
ifg_dem_removed = form_interferogram(primary, secondaries[track], dem_phase[track])
ifg_no_removal  = form_interferogram(primary, secondaries[track], np.zeros_like(dem_phase[track]))
print('interferogram dtype:', ifg_dem_removed.dtype)
print('magnitude max (should respect clip):', float(np.abs(ifg_dem_removed).max()))

## Visual confirmation

The magnitude image should reproduce the scene amplitude (clipped). The phase with the DEM removed should be flatter over the smooth DEM region than the phase formed without removal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
im0 = axes[0].imshow(np.abs(ifg_dem_removed), aspect='auto', origin='lower')
axes[0].set_title('|interferogram| (clipped amp)')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)
im1 = axes[1].imshow(np.angle(ifg_no_removal), cmap='twilight', vmin=-np.pi, vmax=np.pi, aspect='auto', origin='lower')
axes[1].set_title('phase, DEM NOT removed')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)
im2 = axes[2].imshow(np.angle(ifg_dem_removed), cmap='twilight', vmin=-np.pi, vmax=np.pi, aspect='auto', origin='lower')
axes[2].set_title('phase, DEM removed')
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)
for ax in axes:
    ax.set_xlabel('range'); ax.set_ylabel('azimuth')
fig.tight_layout()
plt.show()

## Expected visual outcome

The magnitude panel shows the two planted blobs with magnitude bounded by the amplitude clip. The middle phase panel shows a range-periodic fringe pattern from the un-removed DEM, while the right panel is markedly smoother over the same region, confirming the deramp step removes the reference topographic phase.